In [1]:
import pandas as pd
import numpy as np
import sqlite3

# 1. LOAD BRONZE DATA
# Fix: Using 'r' for Windows paths and adding error handling
try:
    df_cust = pd.read_csv(r'D:\FinTech\Data\Raw\customers.csv')
    df_tx = pd.read_csv(r'D:\FinTech\Data\Raw\transactions.csv')
except FileNotFoundError as e:
    print(f"Error: Could not find the file. {e}")
    # Stop execution if files aren't found
    raise

# --- SILVER TRANSFORMATION: CUSTOMERS ---

# 2. Fix Exact Duplicates & ID Conflicts
df_cust = df_cust.drop_duplicates()
# Fix: Ensure 'signup_date' is datetime so sorting actually works chronologically
df_cust['signup_date'] = pd.to_datetime(df_cust['signup_date'], errors='coerce')
df_cust = df_cust.sort_values('signup_date', ascending=False).drop_duplicates(subset=['cust_id'])

# 3. Standardize Strings
df_cust['full_name'] = df_cust['full_name'].str.strip().str.title()
df_cust['email_address'] = df_cust['email_address'].fillna('unknown@company.com')
df_cust['email_address'] = df_cust['email_address'].str.replace('@@', '@', regex=False)

# 4. Correct Credit Score Outliers
# Use median for realistic fintech imputation
df_cust.loc[(df_cust['credit_score'] < 300) | (df_cust['credit_score'] > 850), 'credit_score'] = np.nan
df_cust['credit_score'] = df_cust['credit_score'].fillna(df_cust['credit_score'].median())

# --- SILVER TRANSFORMATION: TRANSACTIONS ---

# 5. Standardize Categories
df_tx['category'] = df_tx['category'].str.strip().str.capitalize()

# 6. Smart Imputation (Filling Null Amounts)
# Fix: Using 'fillna' inside a transform can sometimes be tricky; this is the robust way:
df_tx['amount'] = df_tx.groupby('category')['amount'].transform(lambda x: x.fillna(x.median()))

# 7. Financial Outlier Handling (Capping)
cap_value = df_tx['amount'].quantile(0.99)
df_tx['amount'] = df_tx['amount'].clip(lower=0, upper=cap_value)

# --- LOAD TO DATA WAREHOUSE (SQL) ---

# 8. Establish Database Connection
conn = sqlite3.connect('FinTech_Warehouse.db')

# 9. Load Cleaned Dataframes to SQL Tables
try:
    df_cust.to_sql('silver_customers', conn, if_exists='replace', index=False)
    df_tx.to_sql('silver_transactions', conn, if_exists='replace', index=False)
    print("🚀 Success: Silver Layer created in FinTech_Warehouse.db")
finally:
    # Always close connection even if the upload fails
    conn.close()

🚀 Success: Silver Layer created in FinTech_Warehouse.db
